# B-tree Splits and the Fill Factor You End Up With

Companion notebook for the [B-tree Mechanics](https://codemia.ai/courses/database_internals/dbi_btree_mechanics) lesson on Codemia.

**What you will build**: a B+tree that counts its own splits, three insert orders run through it,
and the occupancy each one leaves behind. By the end you will be able to say why the same ten
million rows occupy half again as many pages when they arrive in a random order, and why deleting
most of them gives none of that space back.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import bisect

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

CAP = 32        # keys a leaf holds before it has to split
N = 20000       # keys inserted in every experiment below

## Concept recap

A B+tree leaf holds a fixed number of keys. When an insert arrives at a full leaf the leaf splits:
a new page is allocated, half the keys move into it, and a separator is pushed into the parent. If
the parent is full too it splits as well, and in the rare case where that reaches the top the tree
gains a level.

Two consequences follow, and both are visible in a page count rather than in a query plan.

The first is occupancy. A leaf that has just split is half full, and it fills again only if more
keys land in its range. Keys arriving in random order spread across every leaf, so the tree settles
at an average occupancy of around ln 2, which is 69 percent, no matter how many keys you insert.
Keys arriving in ascending order all land in the rightmost leaf, so every split leaves a half empty
page behind that nothing will ever fill again, and the tree settles at 50 percent. Engines detect
the ascending case and split asymmetrically, keeping the full page full and starting a fresh one,
which recovers almost all of the difference.

The second is that deletions do not undo any of this. Removing a key frees a slot in a page; it
does not free the page. A table that was ten gigabytes and is now one gigabyte of live rows is
still ten gigabytes of pages until something rebuilds it.

## A B+tree that counts what it does

In [ ]:
class Node:
    __slots__ = ("leaf", "keys", "kids")

    def __init__(self, leaf):
        self.leaf = leaf
        self.keys = []
        self.kids = []


class BPlusTree:
    """A B+tree with one deliberate simplification: deletes never merge pages.

    That is not a shortcut, it is what the engines in common use actually do. Pages stay
    allocated and partly empty until a rebuild reclaims them.
    """

    def __init__(self, cap=CAP, right_split=False):
        self.cap = cap
        self.right_split = right_split   # split 1 : cap instead of in half when appending
        self.root = Node(True)
        self.splits = 0

    def _descend(self, key):
        node, path = self.root, []
        while not node.leaf:
            i = bisect.bisect_right(node.keys, key)
            path.append((node, i))
            node = node.kids[i]
        return node, path

    def insert(self, key):
        leaf, path = self._descend(key)
        i = bisect.bisect_left(leaf.keys, key)
        if i < len(leaf.keys) and leaf.keys[i] == key:
            return
        leaf.keys.insert(i, key)
        if len(leaf.keys) <= self.cap:
            return
        appended = i == len(leaf.keys) - 1
        mid = self.cap if (self.right_split and appended) else len(leaf.keys) // 2
        right = Node(True)
        right.keys = leaf.keys[mid:]
        leaf.keys = leaf.keys[:mid]
        self.splits += 1
        self._grow(path, leaf, right.keys[0], right)

    def _grow(self, path, left, sep, right):
        while path:
            parent, i = path.pop()
            parent.keys.insert(i, sep)
            parent.kids.insert(i + 1, right)
            if len(parent.keys) <= self.cap:
                return
            mid = len(parent.keys) // 2
            sep = parent.keys[mid]
            sibling = Node(False)
            sibling.keys = parent.keys[mid + 1:]
            sibling.kids = parent.kids[mid + 1:]
            parent.keys = parent.keys[:mid]
            parent.kids = parent.kids[:mid + 1]
            self.splits += 1
            left, right = parent, sibling
        root = Node(False)
        root.keys = [sep]
        root.kids = [left, right]
        self.root = root

    def delete(self, key):
        leaf, _ = self._descend(key)
        i = bisect.bisect_left(leaf.keys, key)
        if i < len(leaf.keys) and leaf.keys[i] == key:
            leaf.keys.pop(i)

    def leaves(self):
        out, stack = [], [self.root]
        while stack:
            node = stack.pop()
            if node.leaf:
                out.append(node)
            else:
                stack.extend(reversed(node.kids))
        return out

    def height(self):
        node, h = self.root, 1
        while not node.leaf:
            node, h = node.kids[0], h + 1
        return h

    def occupancy(self):
        return np.array([len(l.keys) for l in self.leaves()], dtype=float)

In [ ]:
def build(order, cap=CAP, right_split=False):
    tree = BPlusTree(cap=cap, right_split=right_split)
    for k in order:
        tree.insert(int(k))
    return tree


# a correctness check before any conclusions are drawn from it
check = build(rng.permutation(N))
flat = [k for leaf in check.leaves() for k in leaf.keys]
assert len(flat) == N and flat == sorted(flat), "the tree lost or misordered keys"
print("%d keys, %d leaves, height %d, sorted across leaves" %
      (len(flat), len(check.leaves()), check.height()))

The assertion is the part worth keeping. Every number below is read off this structure, so it is
worth proving once that the structure holds every key it was given, in order, before trusting any
of them.

## Three insert orders, one key set

In [ ]:
ascending = np.arange(N)
shuffled = rng.permutation(N)

runs = [
    ("ascending, split in half", build(ascending)),
    ("random, split in half", build(shuffled)),
    ("ascending, split 1 : 32", build(ascending, right_split=True)),
]

print("%-26s %8s %7s %8s %9s" % ("insert order", "leaves", "height", "splits", "fill"))
for name, tree in runs:
    occ = tree.occupancy()
    print("%-26s %8d %7d %8d %8.1f%%" %
          (name, len(occ), tree.height(), tree.splits, 100 * occ.mean() / CAP))

print()
print("ln 2 is %.1f%%, which is where random inserts are expected to settle" % (100 * np.log(2)))

Three numbers to take from that table.

Random inserts land within a point or so of 69 percent, which is the classic result and holds for
any capacity and any number of keys. Ascending inserts with a symmetric split land at almost
exactly 50 percent, because every page except the rightmost one was abandoned the moment it split
and nothing will ever be inserted into it again. The asymmetric split recovers nearly all of that:
the page that is full stays full, the new page starts empty and fills, and the tree ends up close
to 100 percent.

The split counts tell the same story from the other side. The same keys, the same capacity, and
one order needs half again as many pages as another.

## What the occupancy looks like, page by page

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
for ax, (name, tree) in zip(axes, runs):
    occ = tree.occupancy()
    ax.hist(occ, bins=np.arange(0, CAP + 2) - 0.5, color="#5b8def")
    ax.axvline(occ.mean(), color="#d1495b", lw=2)
    ax.set_title("%s\nmean %.1f of %d" % (name, occ.mean(), CAP), fontsize=9)
    ax.set_xlabel("keys in the page")
axes[0].set_ylabel("pages")
plt.tight_layout()
plt.show()

The ascending symmetric case is the one to look at: almost every page holds exactly half of what it
could, in a single spike. That is not an average over a spread, it is every page in the tree being
equally and permanently half empty.

The random case spreads between half full and full, because a page that split can go on receiving
keys. The asymmetric case piles up at the capacity.

## Deleting most of the rows

In [ ]:
tree = build(shuffled)
before_pages, before_fill = len(tree.leaves()), tree.occupancy().mean()

doomed = rng.permutation(N)[: int(0.9 * N)]
for k in doomed:
    tree.delete(int(k))

occ = tree.occupancy()
print("live keys      %7d  ->  %d" % (N, N - len(doomed)))
print("pages          %7d  ->  %d" % (before_pages, len(occ)))
print("mean fill      %7.1f  ->  %.1f  of %d" % (before_fill, occ.mean(), CAP))
print("empty pages    %7d  ->  %d" % (0, int((occ == 0).sum())))
print()
print("a full scan still reads %d pages to return %d rows" % (len(occ), int(occ.sum())))

Ninety percent of the rows are gone and not one page was freed. A sequential scan still reads every
page, because the engine has no way to know in advance which of them are empty, and the tree is now
a tenth as dense as the day it was built.

This is why a table that shrank still reports its old size, why a scan that used to take a second
does not speed up when half the rows are deleted, and why the remedy is a rebuild rather than a
vacuum: reclaiming the slots is cheap and routine, and reclaiming the pages means rewriting the
structure.

## Variations: does the page capacity change any of this

In [ ]:
print("%6s %10s %8s %10s %10s" % ("cap", "leaves", "height", "fill", "splits"))
for cap in [8, 16, 32, 64, 128, 256]:
    t = build(shuffled, cap=cap)
    occ = t.occupancy()
    print("%6d %10d %8d %9.1f%% %10d" %
          (cap, len(occ), t.height(), 100 * occ.mean() / cap, t.splits))

The fill factor barely moves. It is a property of random insertion into a structure that splits in
half rather than of how large the pages are, which is why 69 percent turns up in every engine
regardless of its page size. It drifts below that at the largest capacities here only because twenty
thousand keys no longer make enough pages for the average to settle.

What the capacity does change is the height, and the height is what a point lookup pays. Going from
a capacity of 8 to one of 256 removes levels from the tree, and every level removed is one fewer
page read on every lookup that misses the cache. This is the entire argument for wide pages and
narrow keys: a key twice as wide halves the capacity, which is a height increase waiting to happen.

## What to try next

- Insert in ascending order but with one key in every hundred arriving out of order, which is what a
  timestamp key with clock skew looks like, and see how quickly the asymmetric split stops helping.
- Make the keys variable width, say a random string of 8 to 40 bytes, and let the capacity depend on
  the bytes rather than the count. The fill factor result survives; the height does not.
- Add a fill factor setting that splits before the page is full, leaving room for later updates to
  the rows already there, and measure what it costs in pages to buy that room.
- Rebuild the deleted tree by inserting the surviving keys in ascending order into a fresh tree, and
  compare the page count against what the rebuild reclaimed.